# Store Sales 完整GPU训练入口

启用Kaggle GPU，添加Store Sales竞赛数据，打开网络访问以克隆公开仓库和安装依赖。此入口使用width64、完整起点样本、最多50轮和patience6内层早停；与CPU受限实验明确区分。

需要GPU时才运行本Notebook。没有实际GPU运行回执前，不把它当成已完成的实验。

In [ ]:
import torch,subprocess,sys
from pathlib import Path
assert torch.cuda.is_available(), '请先为当前Kaggle Notebook启用GPU。'
print(torch.__version__,torch.cuda.get_device_name(0))
ROOT=Path('/kaggle/working/store-sales-time-series')
if not ROOT.exists():subprocess.run(['git','clone','https://github.com/Eatproperly/store-sales-time-series.git',str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-advanced-gpu.txt')],check=True)
required=['train.csv','test.csv','stores.csv','holidays_events.csv','oil.csv','transactions.csv','sample_submission.csv']
candidates=[p.parent for p in Path('/kaggle/input').rglob('train.csv') if all((p.parent/f).exists() for f in required)]
assert len(candidates)==1, '请添加Store Sales竞赛数据，并保证只有一套完整输入。'
(ROOT/'data').mkdir(exist_ok=True)
for f in required:
 dest=ROOT/'data'/f
 if not dest.exists():dest.symlink_to(candidates[0]/f)
 else:assert dest.resolve()==(candidates[0]/f).resolve(), '数据目录已有不同文件，需人工核对。'


In [ ]:
# 先检查版本、数据与前向/反向传播，再开始完整GPU实验。
import time
sys.path.insert(0,str(ROOT/'src'))
from store_sales.models.mixer import ForecastNet
model=ForecastNet([54,33,1782,17,5],width=64).cuda()
x=(torch.randn(512,112,8,device='cuda'),torch.randn(512,16,12,device='cuda'),torch.zeros(512,5,dtype=torch.long,device='cuda'),torch.zeros(512,16,device='cuda'))
torch.cuda.synchronize();start=time.perf_counter()
for _ in range(5):model(*x).square().mean().backward();model.zero_grad()
torch.cuda.synchronize();print('GPU seconds per batch:',(time.perf_counter()-start)/5)
del model,x;torch.cuda.empty_cache()


In [ ]:
subprocess.run([sys.executable,'scripts/run_advanced.py','--config','configs/advanced-gpu.yaml','--output-dir','outputs/gpu_full'],cwd=ROOT,check=True)
subprocess.run([sys.executable,'scripts/export_advanced.py','--output-dir','outputs/gpu_full'],cwd=ROOT,check=True)
print('训练与导出完成，成果路径：',ROOT/'Store_Sales_Advanced_Assignment.zip')
